# 01. Data Quality Engine & Telemetry Validation

## Electrical Engineering Context
In electrical utility distribution systems, Supervisory Control and Data Acquisition (SCADA) systems and Advanced Metering Infrastructure (AMI) record thousands of telemetry data points every hour. However, real-world utility data is notoriously vulnerable to:
* **Potential Transformer (PT) / Current Transformer (CT) transducer saturation or open-circuit glitches**
* **Communication packet drops over cellular/GPRS/RF mesh networks**
* **Transducer polarity wiring reversal (leading to negative power factor readings)**
* **Uncalibrated sensors reporting active power $P > S$**

This notebook executes an automated **Data Quality Engine** that audits 100,000+ raw measurements against fundamental electrical boundaries and physical laws.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set paths
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(BASE_DIR) not in sys.path:
    sys.path.append(str(BASE_DIR))

from src.data_validation.validator import DataQualityEngine
from src.utils.config import RAW_DATA_DIR, PROCESSED_DATA_DIR

print(f"Base Directory: {BASE_DIR}")


## 1. Ingesting Raw Distribution Telemetry
We load the raw measurements dataset containing 6 months of 30-minute interval SCADA readings across 12 distribution feeders.


In [ ]:
raw_csv_path = RAW_DATA_DIR / "raw_electrical_measurements.csv"
df_raw = pd.read_csv(raw_csv_path)
print(f"Ingested Raw Records: {len(df_raw):,}")
df_raw.head()


## 2. Executing Data Quality Audit
We evaluate physical boundary checks:
* **Voltage Validity:** $8,000\text{ V} \le V_{avg} \le 14,500\text{ V}$ (11 kV nominal line-to-line)
* **Current Non-Negativity:** $I_{avg} \ge 0\text{ A}$
* **Power Factor Bounds:** $0.50 \le PF \le 1.00$
* **Physics Law:** $P \le S$ ($P = \sqrt{3} V I PF / 1000$)
* **Primary Key Uniqueness:** (feeder_id, timestamp)


In [ ]:
dq_engine = DataQualityEngine()
audit_report = dq_engine.run_audit(df_raw)
report_text = dq_engine.print_report(audit_report)


## 3. Visualizing Telemetry Defect Distributions
We visualize the defect composition and data quality score.


In [ ]:
defect_categories = {
    "Duplicates": audit_report["duplicate_records"],
    "Missing Telemetry": audit_report["missing_record_rows"],
    "Voltage Out-of-Bounds": audit_report["voltage_anomalies"],
    "Current Out-of-Bounds": audit_report["current_anomalies"],
    "Invalid PF (<0.5 or >1)": audit_report["invalid_pf_records"],
    "Physics Inconsistent (P>S)": audit_report["inconsistent_power_records"]
}

plt.figure(figsize=(10, 4.5))
bars = plt.barh(list(defect_categories.keys()), list(defect_categories.values()), color="#e53e3e")
plt.xlabel("Flagged Defective Records Count", fontsize=11, fontweight="bold")
plt.title(f"SCADA Telemetry Defect Breakdown (Overall Quality: {audit_report['overall_data_quality_pct']}%)", fontsize=12, fontweight="bold")
for bar in bars:
    w = bar.get_width()
    plt.text(w + 5, bar.get_y() + bar.get_height()/2, f"{int(w):,}", va="center", fontsize=10)
plt.xlim(0, max(defect_categories.values()) * 1.2)
plt.tight_layout()
plt.show()


## 4. Dataset Cleaning & Physics Re-Derivation
We remove corrupted records, re-derive electrical powers $(S, P, Q)$ using physical equations, and verify 100% clean output.


In [ ]:
df_clean, _ = dq_engine.clean_dataset(df_raw)
print(f"Verified Pristine Output Records: {len(df_clean):,}")
print("Checking for remaining nulls:")
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])
